# HCT Survival Model Testing in Google Colab

This notebook loads the pre-trained HCT survival pipeline used in the project, inspects the model metadata, and runs a small set of reproducible prediction tests.

## What this notebook does
- Installs the dependencies required by the AI service
- Loads the serialized model from `models/trained_pipeline.pkl`
- Shows the model type, feature count, and training summary
- Runs sample patient cases through the pipeline
- Optionally evaluates a CSV batch if you want to test more rows

## How to use in Colab
1. Upload this notebook to Google Colab or open it from the repo after cloning.
2. Run the setup cell first.
3. If the model file is missing, the notebook will try to train from `data/raw/train.csv`.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def run_command(command: str) -> None:
    subprocess.check_call(command, shell=True)

is_colab = 'COLAB_RELEASE_TAG' in os.environ
repo_url = 'https://github.com/SaiLord28/HCT_Survival_Equity_Project.git'

if is_colab:
    workdir = Path('/content/HCT_Survival_Equity_Project')
    if not workdir.exists():
        run_command(f'git clone {repo_url} {workdir}')
else:
    workdir = Path.cwd().resolve()

project_dir = workdir / 'Project' / 'ai_service'
if not project_dir.exists():
    raise FileNotFoundError(f'Could not find ai_service directory at {project_dir}')

sys.path.insert(0, str(project_dir))

requirements_file = project_dir / 'requirements.txt'
if requirements_file.exists():
    run_command(f'pip -q install -r {requirements_file}')

print('Workspace ready')
print(f'Project dir: {project_dir}')
print(f'Running in Colab: {is_colab}')

In [ ]:
import json
import pandas as pd
import numpy as np
from pprint import pprint

from src.pipeline import HCTPipeline

model_path = project_dir / 'models' / 'trained_pipeline.pkl'
data_path = project_dir / 'data' / 'raw' / 'train.csv'

pipeline = HCTPipeline()

if model_path.exists():
    pipeline.load(str(model_path))
    print(f'Loaded trained pipeline from: {model_path}')
else:
    print('Trained model not found. Training a fresh pipeline from train.csv...')
    if not data_path.exists():
        raise FileNotFoundError(f'Could not find training data at {data_path}')
    pipeline.train(str(data_path), model_type='gbm', n_features=45)

print(f'Pipeline trained: {pipeline.is_trained}')
print(f'Model type: {getattr(pipeline.model, 
, 
)}')
print(f'Selected features: {len(pipeline.feature_selector.selected_features)}')
print(f'Group column: {pipeline.group_col}')

training_info = getattr(pipeline, 'training_info', {})
if training_info:
    print('Training stages available:')
    print(list(training_info.get('stages', {}).keys()))

In [ ]:
selected_features = pipeline.feature_selector.selected_features
print('Top selected features:')
for feature_name in selected_features[:20]:
    print('-', feature_name)

if hasattr(pipeline.model, 'get_feature_importance'):
    feature_importance = pipeline.model.get_feature_importance()
    if feature_importance:
        top_importance = sorted(feature_importance.items(), key=lambda item: item[1], reverse=True)[:10]
        print('
Top feature importances:')
        for feature_name, importance in top_importance:
            print(f'- {feature_name}: {importance:.4f}')

if training_info:
    modeling_stage = training_info.get('stages', {}).get('modeling', {})
    if modeling_stage:
        print('
Training metrics:')
        print(json.dumps(modeling_stage.get('train_metrics', {}), indent=2))

## Sample Cases

The following examples are adapted from the project test scripts. They cover low, medium, and high risk profiles so you can quickly verify that the notebook is working end to end.

In [ ]:
sample_patients = [
    {
        'id': 'case_1_low',
        'age_at_hct': 25,
        'donor_age': 28,
        'year_hct': 2023,
        'prim_disease_hct': 'AML',
        'dri_score': 'Low',
        'donor_related': 'Sibling',
        'conditioning_intensity': 'RIC',
        'gvhd_proph': 'TAC+MTX',
        'hla_high_res_8': 8,
        'karnofsky_score': 90,
        'comorbidity_score': 0,
        'tbi_status': 'No TBI',
        'race_group': 'White',
        'ethnicity': 'Not Hispanic or Latino',
        'sex_match': 'M-M'
    },
    {
        'id': 'case_2_medium',
        'age_at_hct': 45,
        'donor_age': 35,
        'year_hct': 2022,
        'prim_disease_hct': 'MDS',
        'dri_score': 'Intermediate',
        'donor_related': 'Unrelated',
        'conditioning_intensity': 'MAC',
        'gvhd_proph': 'TAC+MTX',
        'hla_high_res_8': 7,
        'karnofsky_score': 80,
        'comorbidity_score': 2,
        'tbi_status': 'TBI',
        'race_group': 'White',
        'ethnicity': 'Not Hispanic or Latino',
        'sex_match': 'F-M',
        'cmv_status': '+/-'
    },
    {
        'id': 'case_3_high',
        'age_at_hct': 68,
        'donor_age': 45,
        'year_hct': 2021,
        'prim_disease_hct': 'AML',
        'dri_score': 'High',
        'donor_related': 'Unrelated',
        'conditioning_intensity': 'MAC',
        'gvhd_proph': 'Other',
        'hla_high_res_8': 5,
        'karnofsky_score': 60,
        'comorbidity_score': 5,
        'tbi_status': 'TBI',
        'race_group': 'Black or African-American',
        'ethnicity': 'Not Hispanic or Latino',
        'sex_match': 'F-M',
        'cmv_status': '+/+'
    }
]

patient_frame = pd.DataFrame(sample_patients)
patient_frame[['id', 'age_at_hct', 'prim_disease_hct', 'dri_score', 'race_group']]

In [ ]:
prediction_rows = []
for patient in sample_patients:
    prediction = pipeline.predict(patient)
    prediction_rows.append({
        'patient_id': prediction.patient_id,
        'risk_category': prediction.risk_category,
        'event_probability': prediction.event_probability,
        'confidence_level': prediction.confidence_level,
        'confidence_lower': prediction.confidence_lower,
        'confidence_upper': prediction.confidence_upper,
        'reliability_score': prediction.reliability_score
    })

results_df = pd.DataFrame(prediction_rows).sort_values('event_probability', ascending=False)
results_df.reset_index(drop=True, inplace=True)
results_df

In [ ]:
for patient in sample_patients:
    prediction = pipeline.predict(patient)
    print('=' * 70)
    print(f"Patient: {prediction.patient_id}")
    print(f"Risk category: {prediction.risk_category}")
    print(f"Event probability: {prediction.event_probability:.3f}")
    print(f"Confidence level: {prediction.confidence_level}")
    print(f"Reliability score: {prediction.reliability_score:.3f}")
    if prediction.confidence_lower is not None and prediction.confidence_upper is not None:
        print(f"Confidence interval: [{prediction.confidence_lower:.3f}, {prediction.confidence_upper:.3f}]")
    if prediction.top_risk_factors:
        print('Top risk factors:')
        for factor in prediction.top_risk_factors[:5]:
            print(f"- {factor.get('feature', 'unknown')}: {factor.get('shap_value', 0):.4f}")

output_path = project_dir / 'models' / 'colab_prediction_results.csv'
results_df.to_csv(output_path, index=False)
print(f'
Saved results to: {output_path}')

## Optional Batch Test

If you want to test the pipeline on a larger set, you can run the model directly against the training CSV or any uploaded CSV with the same schema.

In [ ]:
if data_path.exists():
    batch_predictions, batch_summary = pipeline.batch_predict(str(data_path))
    print('Batch summary:')
    pprint(batch_summary)
    print(f'Number of batch predictions: {len(batch_predictions)}')
else:
    print(f'Data file not found: {data_path}')

## Notes

- The notebook prioritizes the saved model in `models/trained_pipeline.pkl`.
- If that file is missing, it falls back to retraining from `data/raw/train.csv`.
- For Colab, the notebook clones the GitHub repository automatically the first time it runs.